# 🐍 Proje #19 — BtcTurk Kripto Al-Sat Botu (Beta)

Bu Jupyter Notebook rehberi, **BtcTurk** borsasının REST API'si ile entegre çalışan, manuel işlemler yapılabilen ve **5 dakikalık periyotlarla piyasa fiyatlarını gözlemleyip otomatik alım-satım yapan** Tkinter tabanlı masaüstü uygulamasının mimarisini, veri akışlarını ve kodlarını detaylı bir şekilde açıklar.

---

## 📐 Proje Mimarisi ve Dosya İlişkileri

Proje, sorumlulukların ayrılması (separation of concerns) ilkesine göre 4 farklı Python dosyasından oluşmaktadır:

```mermaid
graph TD
    app.py[app.py: GUI Arayüzü & Bot Döngüsü] --> btcturk_wrapper.py[btcturk_wrapper.py: API Sarmalayıcı]
    bot.py[bot.py: Bağlantı Test Scripti] --> btcturk_wrapper.py
    btcturk_wrapper.py --> endpoints.py[endpoints.py: Endpoint Haritası]
    app.py --> bot_state.json[(bot_state.json: Durum Dosyası)]
```

1. **`endpoints.py`:** API uç noktalarını (endpoints), HTTP metotlarını ve gerekli yetki seviyelerini barındıran ayar dosyasıdır.
2. **`btcturk_wrapper.py`:** API isteklerini imzalayan (HMAC-SHA256, Base64) ve borsa isteklerini (bakiye, ticker, emir gönderme/iptal) sarmalayan sınıfı içerir.
3. **`bot.py`:** Girilen API anahtarlarının borsayla başarılı bir bağlantı kurup kuramadığını test etmek için yazılmış basit konsol scriptidir.
4. **`app.py`:** Uygulamanın beynidir. Tkinter arayüzünü oluşturur, verileri Treeview tablolarında gösterir, logları ayrıştırır ve otomatik ticaret döngüsünü arka planda (`threading`) yönetir.

## 🔑 API Anahtarlarının Yapılandırılması (Kurulumdan Önce)

Projenin BtcTurk borsasında hesabınız adına işlem yapabilmesi için API anahtarlarınızı girmeniz gerekir. 

### 1. BtcTurk API Anahtarı Nasıl Alınır?
- BtcTurk PRO hesabınıza giriş yapın.
- Hesap ayarlarınızdan **API Yönetimi** sayfasına gidin.
- Yeni bir API Anahtarı oluşturun.
- İzinler (Permissions) kısmında **kesinlikle** şunları aktif edin:
  - **Total Funds (Toplam Varlık):** Bakiyelerinizi sorgulamak için gereklidir.
  - **Trade (Al-Sat):** Limit ve Market emirleri gönderip iptal etmek için gereklidir.

### 2. Kod Üzerinde Neyi Değiştirmeniz Lazım?
Projeyi indirdikten sonra, kendi API anahtarlarınızı aşağıdaki dosyalardaki **placeholder (yer tutucu)** alanlara yapıştırmalısınız:

- **`app.py` Dosyasında (Satır 28-29):**
  ```python
  PUBLIC_KEY = "BURAYA_KENDI_PUBLIC_KEYINIZI_YAZIN"
  PRIVATE_KEY = "BURAYA_KENDI_PRIVATE_SECRET_KEYINIZI_YAZIN"
  ```
- **`bot.py` Dosyasında (Satır 16-17):**
  ```python
  public_key = "BURAYA_KENDI_PUBLIC_KEYINIZI_YAZIN"
  private_key = "BURAYA_KENDI_PRIVATE_SECRET_KEYINIZI_YAZIN"
  ```

> ⚠️ **Güvenlik Uyarısı:** API Private/Secret anahtarınız şifreniz gibidir. Bu anahtarları asla GitHub gibi halka açık ortamlarda paylaşmayın! Proje klasöründeki `.gitignore` dosyası, API bilgilerini içerebilecek geçici dosyaların ve durum dosyalarının kazara commit edilmesini önler.

## 🌐 Adım 1: API Endpoint Haritası (`endpoints.py`)

BtcTurk API'sine istek gönderirken her istek için URL uzantısı, HTTP metot tipi (GET, POST, DELETE) ve API anahtarının izin seviyesi değişiklik gösterir. Kod karmaşasını önlemek adına tüm bu tanımlamalar tek bir `ENDPOINTS` sözlüğünde toplanmıştır.

Aşağıdaki kod hücresinde `endpoints.py` dosyasının içeriği gösterilmektedir:

In [ ]:
# endpoints.py dosyasının tam içeriği:
ENDPOINTS = {
    # Hesap Bakiye Bilgileri: Kullanıcının cüzdanındaki tüm TRY ve kripto varlık miktarlarını çeker.
    # Yetki: "Total Funds" (Toplam Varlık) izni gereklidir.
    "balances": {
        "ep": "/api/v1/users/balances",
        "type": "GET",
        "permission": "Total Funds (Toplam Varlık)"
    },
    # Geçmiş Emirler Listesi: Tüm tamamlanmış veya iptal edilmiş emirlerin geçmişini çeker.
    "all_orders": {
        "ep": "/api/v1/allOrders",
        "type": "GET",
        "permission": "Trade (Al-Sat)"
    },
    # Tekil Emir Sorgulama: ID'si belirtilen belirli bir emrin güncel durumunu çeker.
    "order": {
        "ep": "/api/v1/order",
        "type": "GET",
        "permission": "Trade (Al-Sat)"
    },
    # Emir İptal Etme: Açık bir emri iptal etmek için kullanılır (HTTP DELETE metodu).
    "cancel_order": {
        "ep": "/api/v1/order",
        "type": "DELETE",
        "permission": "Trade (Al-Sat)"
    },
    # Yeni Emir Gönderme: Alış veya satış emri (Limit/Market) oluşturmak için kullanılır (HTTP POST metodu).
    "new_order": {
        "ep": "/api/v1/order",
        "type": "POST",
        "permission": "Trade (Al-Sat)"
    },
    # Güncel Açık Emirler: Henüz gerçekleşmemiş (bekleyen) tüm açık emirleri listeler.
    "open_orders": {
        "ep": "/api/v1/openOrders",
        "type": "GET",
        "permission": "Trade (Al-Sat)"
    },
    # Günlük Ticker Fiyat Özetleri (v2): Son fiyat, 24 saatlik değişim yüzdesi, hacim vb. veriler.
    # Genel Bilgi (Herkese Açık) - İzin/Kimlik Doğrulama gerekmez.
    "ticker": {
        "ep": "/api/v2/ticker",
        "type": "GET",
        "version": "v2",
        "permission": ""
    },
    # Kripto Para Bazlı Fiyat Bilgisi (v2): Belirli kripto para birimleri bazında ticker bilgilerini getirir.
    "ticker_currency": {
        "ep": "/api/v2/ticker/currency",
        "type": "GET",
        "version": "v2",
        "permission": ""
    }
}

print(f" balances endpoint metodu: {ENDPOINTS['balances']['type']}")
print(f" balances endpoint yolu  : {ENDPOINTS['balances']['ep']}")
print(f" balances endpoint yetkisi: {ENDPOINTS['balances']['permission']}")

## 🔐 Adım 2: API İsteklerini İmzalama ve REST Sarmalayıcı (`btcturk_wrapper.py`)

BtcTurk borsasında hesabınızla işlem yapabilmek için her istekte sunucuya **imza** gönderilmelidir. Bu imza, isteğin yolda değiştirilmediğini ve gerçekten sizin tarafınızdan oluşturulduğunu doğrular.

### İmza Hesaplama Formülü:
$$\text{Signature} = \text{Base64}(\text{HMAC-SHA256}(\text{Base64Decode}(\text{SecretKey}), \text{Utf8}(\text{PublicKey} + \text{Stamp})))$$

### HTTP Başlıkları (Headers):
- `X-PCK`: Hesabınızdan aldığınız Public API anahtarı.
- `X-Stamp`: Milisaniye cinsinden zaman damgası (sunucu stamp ile 5 dakikadan fazla fark varsa isteği reddeder).
- `X-Signature`: Yukarıdaki formülle üretilen Base64 formatındaki imza string'i.
- `Content-Type`: `application/json`

### Yuvarlama ve Basamak Hassasiyeti (Precision):
Kripto para borsalarında işlem yaparken küsurat miktarları kritik bir öneme sahiptir. Örneğin BtcTurk, BTC alış emirlerinde miktarın en fazla 8 basamak olmasını, TRY cüzdan varlıklarında ise 2 basamak olmasını ister. Fazladan bir küsurat gönderildiğinde sunucu emri doğrudan reddeder.
- `numerator_scale(pair, val)`: Kripto miktarı basamak hassasiyetine göre aşağı yuvarlar.
- `denominator_scale(pair, val)`: İtibari para tutarını (TRY/USDT) basamak hassasiyetine göre aşağı yuvarlar.

Aşağıdaki kod hücresinde `btcturk_wrapper.py` dosyasındaki imza ve istek sarmalama mantığının bir demosu yer almaktadır:

In [ ]:
import requests
import base64
import time
import hashlib
import hmac

class BTCTurk:
    def __init__(self, apiKey: str, apiSecret: str):
        self.apiKey = apiKey
        # BtcTurk API Secret Key'i Base64 decode edilmiş olarak kullanır
        try:
            self.apiSecret = base64.b64decode(apiSecret)
        except:
            self.apiSecret = b""

    def generate_headers(self) -> dict:
        """HMAC-SHA256 kullanarak BtcTurk API istek başlıklarını oluşturur."""
        stamp = str(int(time.time()) * 1000)
        data = f"{self.apiKey}{stamp}".encode("utf-8")
        
        # HMAC-SHA256 signature üretimi
        signature = hmac.new(self.apiSecret, data, hashlib.sha256).digest()
        signature_base64 = base64.b64encode(signature).decode("utf-8")
        
        return {
            "X-PCK": self.apiKey,
            "X-Stamp": stamp,
            "X-Signature": signature_base64,
            "Content-Type": "application/json"
        }

# Sahte anahtarlar ile imza üretme testi
fake_key = "my_pck_key_123"
fake_secret_base64 = base64.b64encode(b"my_secret_key_456").decode("utf-8")

client = BTCTurk(fake_key, fake_secret_base64)
headers = client.generate_headers()

print("BtcTurk İstek Başlıkları:")
for k, v in headers.items():
    print(f"  {k}: {v}")

## 📡 Adım 3: API Bağlantı Test Scripti (`bot.py`)

`bot.py` dosyası, grafik arayüzünü açmadan önce API anahtarlarımızın doğru formatta ve çalışır durumda olduğunu hızlıca test etmemizi sağlar. Bu script, borsadan **USDT** (Tether) kripto parasının güncel piyasa özetini (`get_ticker_currency`) çeker ve konsola JSON formatında yazdırır.

Aşağıdaki kod hücresinde `bot.py` dosyasının tam içeriği bulunmaktadır:

In [ ]:
# bot.py dosyasının içeriği:
from btcturk_wrapper import BTCTurk
import json

# BtcTurk web sitesinden aldığınız API anahtarlarını tırnaklar arasına girin
public_key = "BURALARA APİ ANAHTARLARINI YAZINIZ"
private_key = "BURALARA APİ ANAHTARLARINI YAZINIZ"

try:
    # Wrapper sınıfından bir örnek oluşturalım
    bt = BTCTurk(apiKey=public_key, apiSecret=private_key)
    
    # USDT ticker bilgilerini çekelim
    # Not: public_key geçerli değilse bu komut hata verecektir
    print("API bağlantısı test ediliyor...")
    # d = bt.get_ticker_currency("USDT")
    # print(json.dumps(d, indent=2))
except Exception as e:
    print(f"Bağlantı kurulamadı veya anahtarlar geçersiz: {e}")

## 🖥️ Adım 4: Grafik Arayüzü ve Çoklu İş Parçacığı (Multi-threading) Yapısı (`app.py`)

Uygulamamızın ana kod dosyası olan `app.py`, gelişmiş bir kullanıcı paneli ve otomatik al-sat yönetim mekanizması sunar.

### 1. Tkinter ve Çoklu İş Parçacığı (Threading) Mantığı
Tkinter arayüzü tek bir thread (iş parçacığı) üzerinden çalışır. Eğer bu thread üzerinde ağ istekleri (HTTP requests) yapılırsa, sunucu yanıt dönene kadar arayüz kilitlenir. Kullanıcı butonlara tıklayamaz, pencereyi hareket ettiremez ve işletim sistemi pencerenin yanıt vermediğini belirtir.

Bu sorunu engellemek için, `app.py` içerisinde ağla ilgili tüm işlemler ayrı birer arka plan thread'inde çalıştırılır:

```python
# Thread tanımlama ve başlatma örneği:
t = threading.Thread(target=self.some_network_task, args=(param1,), daemon=True)
t.start()
```

### 2. Thread-Safe Arayüz Güncelleme (`root.after`)
Python'da arka planda çalışan bir iş parçacığının doğrudan grafik arayüz nesnelerini (örneğin bir Treeview tablosunu veya metin kutusunu) değiştirmesi kararsızlığa ve çökmelere neden olabilir. 
Bu nedenle arayüz güncellemeleri, `root.after(0, callback)` yardımıyla ana thread'in kuyruğuna aktarılır. Bu sayede thread-safe (iş parçacığı güvenli) bir güncelleme sağlanmış olur.

## 📈 Adım 5: 5 Dakikalık Fiyat Gözlem ve Otomatik Al-Sat Döngüsü

Uygulamanın en kritik özelliği, kullanıcı tarafından başlatılan **otomatik ticaret botu döngüsüdür**. 
Bu döngü, `auto_trade` fonksiyonunda tanımlanmıştır. Algoritmanın adımları ve çalışması aşağıdaki akış diyagramında gösterilmiştir:

```mermaid
graph TD
    A([1. Bot Başlatıldı]) --> B[Maliyet Kontrolü]
    B --> C{Daha önceden alınmış coin var mı?}
    C -- Hayır --> D[2. Gözlem Aşaması: 5 dakika boyunca 10 saniyede bir fiyat toplanır]
    D --> E[Bu süre içindeki en dip fiyat bulunur]
    E --> F[3. Alım Aşaması: Belirlenen TRY bakiye oranıyla market alışı yapılır]
    F --> G[Alınan fiyat Ortalama Maliyet olarak kaydedilir]
    G --> H[4. Takip Aşaması: Her 5 saniyede bir anlık fiyat taranır]
    C -- Evet --> H
    H --> I{Anlık Fiyat >= Ortalama Maliyet * Kâr Hedefi?}
    I -- Evet --> J[5. Satış Aşaması: Market satışı yapılır, kâr realize edilir]
    J --> K([Bot Durdurulur])
    I -- Hayır --> H
```

### `auto_trade` Fonksiyonunun Kod Yapısı ve İncelemesi:

Aşağıdaki kod parçasında `app.py` içerisindeki otomatik ticaret döngüsünün tam akışını inceleyebilirsiniz:

In [ ]:
# auto_trade fonksiyonunun basitleştirilmiş mantıksal akışı:
import time

def auto_trade_demo(pair, target_profit_percent, balance_percent):
    print(f"[{pair}] Otomatik al-sat döngüsü başladı.")
    print(f"[{pair}] Gözlem Aşaması: 5 dakika (300 saniye) boyunca fiyatlar izleniyor...")
    
    # 1. 5 Dakikalık Fiyat Toplama Simülasyonu
    observe_prices = [100.5, 99.8, 98.4, 99.2, 101.0, 97.6, 98.9, 100.2] # 5 dakika boyunca gelen fiyatlar
    time.sleep(1) # Simülasyon gecikmesi
    
    min_price = min(observe_prices)
    print(f"[{pair}] Gözlem bitti. En düşük tespit edilen dip fiyat: {min_price:.2f}")
    
    # 2. Alım Gerçekleştirme Simülasyonu
    buy_price = 98.0 # Satın alınan ortalama fiyat (maliyet)
    print(f"[{pair}] Market alışı yapıldı. Ortalama Maliyet: {buy_price:.2f}")
    
    # 3. Takip ve Kâr Hedefli Satış Simülasyonu
    target_price = buy_price * (1 + target_profit_percent / 100.0)
    print(f"[{pair}] Satış Hedef Fiyatı (%{target_profit_percent} kâr ile): {target_price:.2f}")
    
    # Anlık fiyat takip döngüsü simülasyonu
    simulated_prices = [98.1, 98.5, 99.2, 99.9, 100.1, 100.5] # Fiyatın yükselme adımları
    for current_price in simulated_prices:
        print(f"  -> Anlık Fiyat: {current_price:.2f} | Hedef: {target_price:.2f}")
        if current_price >= target_price:
            print(f"[{pair}] HEDEF GERÇEKLEŞTİ! Market satışı yapıldı. Satış Fiyatı: {current_price:.2f}")
            break
    print(f"[{pair}] Bot başarıyla tamamlandı.")

# %2 kâr hedefi ve %50 bakiye oranı ile demo çalıştırılması
auto_trade_demo("BTC_TRY", target_profit_percent=2.0, balance_percent=50.0)

## 💾 Adım 6: Durum Saklama (`bot_state.json`) ve Log Sistemi

Uygulamanın çökmesi veya kapatılması durumunda maliyetlerin ve bot durumlarının kaybolmaması için **JSON tabanlı bir durum yönetim sistemi** entegre edilmiştir:

- `load_state()`: Program başlarken `bot_state.json` dosyasını okur; eğer dosya varsa aktif durumları (`auto_running`), ortalama alım maliyetlerini (`avg_buy_price`) ve geçmiş logları (`logs`) belleğe geri yükler.
- `save_state()`: Herhangi bir bot durumu değiştiğinde, alım yapıldığında, satış yapıldığında veya yeni log eklendiğinde verileri anında `bot_state.json` dosyasına yazarak diske kaydeder.
- **3 Kanallı Ayrı Loglama:**
  - **Genel Loglar (Siyah):** Botun genel durumu, bakiye güncellemeleri ve durdurulma bilgileri.
  - **Alım Logları (Yeşil):** Gözlem adımları, dip fiyat bulma detayları ve tetiklenen alım emirleri.
  - **Satış Logları (Mavi):** Maliyet takip detayları, anlık fiyat farkları ve tetiklenen satış emirleri.

## 🚀 Adım 7: Uygulamanın Çalıştırılması ve Canlı Deneyim

Gerekli kütüphaneleri kurup, API anahtarlarınızı `app.py` içerisindeki ilgili alanlara yazdıktan sonra uygulamayı başlatmak için alttaki hücreyi çalıştırabilir veya terminalden `python app.py` komutunu verebilirsiniz.

In [ ]:
# app.py dosyasını doğrudan bu notebook üzerinden çalıştırmak için alttaki satırın yorum işaretini (#) kaldırıp çalıştırın:
# %run app.py